# AI Job Displacement Analysis

LLM-Based Assessment System

# API Key

Google Gemini: https://ai.google.dev/

# Imports

In [13]:
import json
import asyncio
import pandas as pd
from google import genai
from google.genai import types
from pydantic import BaseModel
from typing import Dict, List, Any

# Response Schema

In [14]:
class DimensionScore(BaseModel):
    reasoning: str
    score: int

class TaskAssessment(BaseModel):
    task_predictability: DimensionScore
    interaction_medium: DimensionScore
    social_requirement: DimensionScore
    environmental_stability: DimensionScore
    consequence_of_failure: DimensionScore
    regulatory_barrier: DimensionScore
    economic_arbitrage: DimensionScore

# Gemini Automation Pipeline Class

In [15]:
class AutomationAssessmentPipeline:
    def __init__(self, api_key: str, data_path: str):
        self.client = genai.Client(api_key=api_key)
        self.data_path = data_path
        self.job_data = self._load_json_data()

    def _load_json_data(self) -> List[Dict[str, Any]]:
        """Reads the prepared JSON dataset."""
        with open(self.data_path, 'r') as file:
            return json.load(file)

    def build_assessment_prompt(self, task_description: str, job_context: str) -> str:
        """Constructs prompt matching the exact framework prompt specification."""
        return f"""Analyze the following job task and evaluate its automation potential across 7 distinct dimensions.
For each dimension, assign a score from 1 to 5, where:
1 = Very difficult to automate (Strong human advantage / External barrier)
5 = Highly automatable (Strong AI/Machine advantage / No barrier)

Job Context: "{job_context}"
Task to Analyze: "{task_description}"

Dimensions to evaluate (Score 1-5):
1. Task Predictability: Is the internal logic heuristic/intuition-based (1) or strictly algorithmic and rule-based (5)?
2. Interaction Medium: Does the task require complex physical manipulation (1) or is it purely abstract digital processing (5)?
3. Social Requirement: Does the task demand deep empathy and trust-building (1) or can it be executed in total isolation (5)?
4. Environmental Stability: Does the work occur in a highly chaotic/unpredictable environment (1) or a perfectly engineered static environment (5)?
5. Consequence of Failure: Are the stakes of a mistake catastrophic (1) or trivial and easily reversed (5)?
6. Regulatory Barrier: Does the law strictly mandate a certified human (1) or is the output completely unregulated (5)?
7. Economic Arbitrage: Is human labor so cheap that automation ROI is negligible (1) or is human labor highly expensive making ROI massive (5)?

Provide your assessment STRICTLY in the following JSON format. Be specific and justify each score in the reasoning field BEFORE providing the score to ensure sound logic.

{{
  "task_predictability": {{"reasoning": "...", "score": X}},
  "interaction_medium": {{"reasoning": "...", "score": X}},
  "social_requirement": {{"reasoning": "...", "score": X}},
  "environmental_stability": {{"reasoning": "...", "score": X}},
  "consequence_of_failure": {{"reasoning": "...", "score": X}},
  "regulatory_barrier": {{"reasoning": "...", "score": X}},
  "economic_arbitrage": {{"reasoning": "...", "score": X}}
}}"""

    async def assess_task_automation(self, task_description: str, job_context: str) -> Dict[str, Any]:
        """Calls Gemini API via AsyncChat to structure response."""
        prompt = self.build_assessment_prompt(task_description, job_context)
        max_retries = 3
        backoff_factor = 15.0
        
        for attempt in range(max_retries):
            try:
                chat = self.client.aio.chats.create(
                    model='gemini-3.6-flash',
                    config=types.GenerateContentConfig(
                        temperature=0.2,
                        response_mime_type="application/json",
                        response_schema=TaskAssessment,
                        system_instruction="You are an expert in labor economics and AI automation."
                    )
                )
                
                response = await chat.send_message(prompt)
                return json.loads(response.text)
                
            except Exception as e:
                error_str = str(e)
                if "429" in error_str or "RESOURCE_EXHAUSTED" in error_str:
                    wait_time = backoff_factor * (attempt + 1)
                    print(f"Rate limit reached. Pausing for {wait_time}s (Attempt {attempt+1}/{max_retries})...")
                    await asyncio.sleep(wait_time)
                else:
                    print(f"Error evaluating task '{task_description[:30]}...': {e}")
                    break
        return {}

    def calculate_automation_score(self, dimension_scores: Dict[str, Any], weights: Dict[str, float]) -> float:
        """Calculates normalized overall automation potential score (0 to 100%)."""
        overall_score = 0.0
        for dim, details in dimension_scores.items():
            if dim in weights and isinstance(details, dict) and 'score' in details:
                normalized_score = (details['score'] / 5.0) * 100
                overall_score += normalized_score * weights[dim]
        return overall_score

    async def process_all_occupations(self, weights: Dict[str, float]) -> List[Dict[str, Any]]:
        """Processes occupations iteratively without artificial delays."""
        results = []
        for job in self.job_data:
            job_title = job.get('job_title', 'Unknown')
            job_desc = job.get('job_description', '')
            job_context = f"{job_title} - {job_desc}"
            
            print(f"\nProcessing Occupation: {job_title}")
            
            occupation_task_results = []
            tasks = job.get('tasks', [])
            total_tasks = len(tasks)
            
            for idx, task in enumerate(tasks, start=1):
                task_desc = task['task_description']
                importance = task.get('importance')
                
                # Assign default weight of 1.0 for any tasks with missing or null importance data
                importance_val = float(importance) if importance is not None else 1.0
                
                print(f"  -> Evaluating task ({idx}/{total_tasks}): {task_desc[:40]}...")
                assessment = await self.assess_task_automation(task_desc, job_context)
                
                if assessment:
                    task_score = self.calculate_automation_score(assessment, weights)
                    occupation_task_results.append({
                        "job_title": job_title,
                        "task_id": task['task_id'],
                        "task_description": task_desc,
                        "dimension_assessments": assessment,
                        "task_automation_score": task_score,
                        "importance": importance_val
                    })
                
            # Calculate the weighted Occupation Automation Score
            if occupation_task_results:
                total_weighted_score = sum(res['task_automation_score'] * res['importance'] for res in occupation_task_results)
                total_importance = sum(res['importance'] for res in occupation_task_results)
                occ_automation_score = total_weighted_score / total_importance if total_importance > 0 else 0.0
                
                # Append final structured layout matching the desired df_results output
                for res in occupation_task_results:
                    results.append({
                        "job_title": res["job_title"],
                        "task_id": res["task_id"],
                        "task_description": res["task_description"],
                        "dimension_assessments": res["dimension_assessments"],
                        "task_automation_score": res["task_automation_score"],
                        "occ_automation_score": round(occ_automation_score, 2)
                    })
                
        return results

# Configuration & Weights

In [16]:
# Load API Key from local text file
with open("../api_key.txt", "r") as f:
    API_KEY = f.read().strip()

DATA_PATH = "../data/processed/unified_data.json"

# Exact weight distribution from framework specification
dimension_weights = {
    "task_predictability": 0.15,
    "interaction_medium": 0.15,
    "social_requirement": 0.15,
    "environmental_stability": 0.15,
    "consequence_of_failure": 0.15,
    "regulatory_barrier": 0.15,
    "economic_arbitrage": 0.10
}

# Initialize Pipeline

In [17]:
pipeline = AutomationAssessmentPipeline(
    api_key=API_KEY,
    data_path=DATA_PATH
)

# Test Variance

In [18]:
target_job = "Telemarketers" 
pipeline.job_data = [job for job in pipeline.job_data if job.get("job_title") == target_job]

if not pipeline.job_data:
    print(f"Occupation '{target_job}' not found in dataset.")
else:
    iterations = 5
    all_results = []
    
    # Run the evaluation multiple times
    for i in range(iterations):
        print(f"Running iteration {i+1}/{iterations}...")
        
        # Await the pipeline evaluation
        run_assessments = await pipeline.process_all_occupations(dimension_weights)
        
        if run_assessments:
            # Capture task-level data alongside the occupation score
            for task_result in run_assessments:
                all_results.append({
                    "iteration": i + 1,
                    "job_title": target_job,
                    "task_id": task_result["task_id"],
                    "task_description": task_result["task_description"],
                    "task_automation_score": task_result["task_automation_score"],
                    "occ_automation_score": task_result["occ_automation_score"]
                })
    
    # Consolidate into a DataFrame to analyze score stability
    df_variance = pd.DataFrame(all_results)
    
    # Group by job to view min, max, mean, standard deviation, and variance across iterations
    occ_constancy_report = df_variance.groupby("job_title")["occ_automation_score"].agg(
        ['min', 'max', 'mean', 'std']
    ).round(2).reset_index()
    
    # Group by task ID and description to view task-level variance
    task_constancy_report = df_variance.groupby(["task_id", "task_description"])["task_automation_score"].agg(
        ['min', 'max', 'mean', 'std']
    ).round(2).reset_index()
    
    print("\n--- Occupation Automation Score Variance Report ---")
    display(occ_constancy_report)
    
    print("\n--- Task Automation Score Variance Report ---")
    display(task_constancy_report)

Running iteration 1/5...

Processing Occupation: Telemarketers
  -> Evaluating task (1/12): Contact businesses or private individual...
  -> Evaluating task (2/12): Obtain customer information such as name...
  -> Evaluating task (3/12): Explain products or services and prices,...
  -> Evaluating task (4/12): Record names, addresses, purchases, and ...
  -> Evaluating task (5/12): Maintain records of contacts, accounts, ...
  -> Evaluating task (6/12): Answer telephone calls from potential cu...
  -> Evaluating task (7/12): Deliver prepared sales talks, reading fr...
  -> Evaluating task (8/12): Telephone or write letters to respond to...
  -> Evaluating task (9/12): Adjust sales scripts to better target th...
  -> Evaluating task (10/12): Obtain names and telephone numbers of po...
  -> Evaluating task (11/12): Schedule appointments for sales represen...
  -> Evaluating task (12/12): Conduct client or market surveys to obta...
Running iteration 2/5...

Processing Occupation: Telemarke

,job_title,min,max,mean,std
0,Telemarketers,86.92,88.48,87.48,0.54



--- Task Automation Score Variance Report ---


,task_id,task_description,min,max,mean,std
0,4618,"Deliver prepared sales talks, reading from scr...",80.0,86.0,84.2,2.68
1,4619,Contact businesses or private individuals by t...,80.0,83.0,81.8,1.64
2,4620,"Explain products or services and prices, and a...",80.0,83.0,82.4,1.34
3,4621,"Obtain customer information such as name, addr...",89.0,92.0,91.4,1.34
4,4622,"Record names, addresses, purchases, and reacti...",90.0,95.0,93.6,2.19
5,4623,Adjust sales scripts to better target the need...,89.0,92.0,90.8,1.64
6,4624,Obtain names and telephone numbers of potentia...,95.0,98.0,96.0,1.41
7,4625,Answer telephone calls from potential customer...,81.0,83.0,82.2,1.10
8,4626,Telephone or write letters to respond to corre...,81.0,83.0,82.6,0.89
9,4627,"Maintain records of contacts, accounts, and or...",93.0,95.0,94.6,0.89


# All Occupations

In [20]:
pipeline.job_data = pipeline._load_json_data() 
final_assessments = await pipeline.process_all_occupations(dimension_weights)


Processing Occupation: Software Developers
  -> Evaluating task (1/17): Analyze user needs and software requirem...
  -> Evaluating task (2/17): Develop or direct software system testin...
  -> Evaluating task (3/17): Confer with systems analysts, engineers,...
  -> Evaluating task (4/17): Modify existing software to correct erro...
  -> Evaluating task (5/17): Prepare reports or correspondence concer...
  -> Evaluating task (6/17): Analyze information to determine, recomm...
  -> Evaluating task (7/17): Store, retrieve, and manipulate data for...
  -> Evaluating task (8/17): Design, develop and modify software syst...
  -> Evaluating task (9/17): Determine system performance standards....
  -> Evaluating task (10/17): Consult with customers or other departme...
  -> Evaluating task (11/17): Confer with data processing or project m...
  -> Evaluating task (12/17): Monitor functioning of equipment to ensu...
  -> Evaluating task (13/17): Coordinate installation of software syst...
  ->

# Results

In [ ]:
df_results = pd.DataFrame(final_assessments)
df_results.to_json("../src/automation_assessments.json", orient="records", indent=4)
df_results.head()

,job_title,task_id,task_description,dimension_assessments,task_automation_score,occ_automation_score
0,Software Developers,21662,Analyze user needs and software requirements t...,{'task_predictability': {'reasoning': 'Analyzi...,65.0,75.71
1,Software Developers,21669,Develop or direct software system testing or v...,{'task_predictability': {'reasoning': 'Develop...,82.0,75.71
2,Software Developers,21664,"Confer with systems analysts, engineers, progr...",{'task_predictability': {'reasoning': 'Conferr...,73.0,75.71
3,Software Developers,21670,"Modify existing software to correct errors, ad...",{'task_predictability': {'reasoning': 'Modifyi...,82.0,75.71
4,Software Developers,21673,Prepare reports or correspondence concerning p...,{'task_predictability': {'reasoning': 'Prepari...,91.0,75.71


Drop Janitors and Cleaners occupation, all task were not analyzed

Pipeline runtime needs to be reduced

In [ ]:
file_path = '../src/automation_assessments.json'

# Read the JSON file
with open(file_path, 'r') as file:
    job_data = json.load(file)

# Filter out "Janitors and Cleaners"
filtered_data = [job for job in job_data if job.get("job_title") != "Janitors and Cleaners"]

# Save the updated data back to the file
with open(file_path, 'w') as file:
    json.dump(filtered_data, file, indent=4)